# Aula: IDHM por Unidade da Federação
Lendo `Tabela4.csv` (separador `;`, decimal `,`) e plotando com matplotlib.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CSV = "Tabela4.csv"  # ajuste o caminho se precisar

## 1) Abrir o CSV
O título ocupa a primeira linha, então o cabeçalho real começa na linha 2 (`skiprows=1`).

In [ ]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="latin-1", skiprows=1)
df.head()

## 2) Limpar colunas e linhas vazias

In [ ]:
df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
df.columns = [str(c).strip() for c in df.columns]
anos = [c for c in df.columns if str(c).isdigit()]
print(anos)
df.shape

## 3) Ordenar os estados pelo maior IDH em 2024

In [ ]:
df.sort_values("2024", ascending=False)[["Sigla", "Estado", "2024"]]

## 4) Qual estado melhorou mais entre 1991 e 2024?

In [ ]:
df["Melhora"] = df["2024"] - df["1991"]
df.sort_values("Melhora", ascending=False)[["Estado", "1991", "2024", "Melhora"]]

## 5) Algum estado piorou?

In [ ]:
df[df["Melhora"] < 0][["Estado", "1991", "2024", "Melhora"]]
# vazio = nenhum estado piorou no período completo

## 6) Do formato largo para o longo (`melt`)

In [ ]:
id_vars = [c for c in df.columns if c not in anos and c != "Melhora"]
df_longo = df.drop(columns=["Melhora"]).melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo.head()

### Quedas pontuais de um ano para o outro (ex.: 2021)

In [ ]:
df_longo = df_longo.sort_values(["Sigla", "Ano"])
df_longo["Var"] = df_longo.groupby("Sigla")["IDH"].diff()
quedas = df_longo[df_longo["Var"] < 0]
quedas.groupby("Sigla").size().sort_values(ascending=False)

## 7) Apenas Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mg["Ano"], mg["IDH"], marker="o")
ax.set_title("Evolução do IDH - Minas Gerais (1991-2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.grid(alpha=.3)
fig.tight_layout()
plt.show()

## 8) Todos os estados

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()